# Phase 12.9 — Search Statistical Analysis

Load Phase 12.5 / 12.7 evaluation artifacts, extract paired per-query metrics, and inspect bootstrap CIs and permutation p-values.
No retrieval, ranking, or metric recomputation.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.retrieval.evaluation.search_evaluation.statistical_artifact import (
    validate_search_statistical_analysis_artifact,
)
from productiq.retrieval.evaluation.search_evaluation.statistical_runner import (
    build_default_statistical_configuration,
    run_search_statistical_analysis,
)
from productiq.retrieval.evaluation.search_evaluation.statistical_schema import (
    SEARCH_STATISTICAL_ANALYSIS_FILENAME,
    SearchStatisticalMetricName,
)

In [ ]:
config = build_default_statistical_configuration()
result = run_search_statistical_analysis(ROOT, configuration=config)
print("comparisons:", len(result.comparisons))
print("benchmark:", result.lineage.benchmark_name, result.lineage.benchmark_version)
print("pairs configured:", len(result.configuration.comparison_pairs))
for note in result.provenance_notes:
    print("note:", note)

In [ ]:
mrr_rows = [r for r in result.comparisons if r.metric_name == SearchStatisticalMetricName.MRR]
for row in mrr_rows[:6]:
    print(
        row.reference_variant_name,
        "vs",
        row.candidate_variant_name,
        "delta",
        row.observed_delta,
        "p",
        row.p_value,
        "n",
        row.sample_size,
    )

In [ ]:
p10 = [
    r
    for r in result.comparisons
    if r.metric_name == SearchStatisticalMetricName.PRECISION_AT_K and r.k == 10
]
for row in p10:
    ci = row.confidence_interval
    print(
        row.candidate_variant_name,
        "P@10 delta",
        row.observed_delta,
        "CI",
        (ci.lower, ci.upper) if ci else None,
    )

In [ ]:
from productiq.retrieval.evaluation.search_evaluation.statistical_artifact import (
    build_search_statistical_analysis_artifact_dict,
    write_search_statistical_analysis_artifact,
)

path = ROOT / "resources" / "evaluation" / SEARCH_STATISTICAL_ANALYSIS_FILENAME
write_search_statistical_analysis_artifact(
    path,
    build_search_statistical_analysis_artifact_dict(result),
)
payload = json.loads(path.read_text(encoding="utf-8"))
validate_search_statistical_analysis_artifact(payload)
print("checksum:", payload["deterministic_checksum_sha256"])
print("artifact:", path)